In [ ]:
#Load packages
import os
import shutil
import nd2reader
import pims
import numpy as np
import matplotlib.pyplot as plt
from skimage import measure
import pandas as pd
from matplotlib.collections import LineCollection
from datetime import datetime

In [ ]:
###Select dataset to analyze
input_path = 'D:/Augusto/Images/NSPARC/FISH-Drugs/20250727-NSPARC/TNF-NKF-DMSO' #avoid writing a dash / at the end
input_file = 'HEK293FT-TNF-NKF-DMSO_0002.nd2'

#Define which channel corresponds to fluorescent protein and which one corresponds to nucleus marker ('fp' or 'nucleus')
ch1 = 'nucleus'
ch2 = 'fp'
ch3 = 'cy3'
ch4 = 'cy5'

###Select associated mask
mask_path = 'D:/Augusto/Images/NSPARC/FISH-Jun2025/Analysis/CellSegmentation/CellposeMasks' #avoid writing a dash / at the end
mask_file = '20250727_HEK293FT-TNF-NKF-DMSO_0002-m3d_cp_masks.tif'

###Select FISHPairs.txt file
fishPairs_path = 'D:/Augusto/Images/NSPARC/FISH-Jun2025/Analysis/SpotLocation'
fishPairs_file = '20250727_HEK293FT-TNF-NKF-DMSO_0002_fish.txt'

print("Analysis running")

# print(os.listdir(path))
# datafile = os.path.join(path, file)
# print(datafile)

In [ ]:
###check that all input files (image, fish spots, and masks) match

input_file_root = input_path.split('/')[-2].split('-')[-2] + "_" + input_file.split('.')[-2]
fishPairs_file_root = fishPairs_file.split("_fish.txt")[0]
#mask_file_root = mask_file.split("_cp_masks.tif")[0]
mask_file_root = mask_file.split("-m3d_cp_masks.tif")[0] #if some filtered was used for masks

print(input_file_root)
print(fishPairs_file_root)
print(mask_file_root)

if (input_file_root == fishPairs_file_root == mask_file_root):
    print("Yey! input_file, fishPairs, and mask_file names match")
else:
    print("Warning: input_file, fishPairs, and mask_file names do not match!")
    raise ValueError("Are you sure your inputs are correct?.")




In [ ]:
###Set output path and file
output_path = 'D:/Augusto/Images/NSPARC/FISH-Jun2025/Analysis/RadialResults/'

input_date = input_path.split('/')[-2].split('-')[-2]
output_name = input_date + "_" + input_file.split('.')[-2] + "_radial"

output_file = output_path + output_name + '.txt'
output_file_supplement = output_path + output_name + '_supplement.txt'

print('output_file', output_file)
print('output_file_supplement', output_file_supplement)



In [ ]:
###Generate folder to save images

output_plot_folder_name = input_date + "_" + input_file.split('.')[-2]
output_plot_folder = output_path + output_plot_folder_name

print('output_plot_folder', output_plot_folder)

if os.path.exists(output_plot_folder): # Check if the folder exist
    shutil.rmtree(output_plot_folder) #Delete the folder
    print("Folder already exists, folder deleted.")
    os.makedirs(output_plot_folder) #Create the folder
    print("Folder created successfully!")
else:
    os.makedirs(output_plot_folder)
    print("Folder created successfully!") #Create the folder
    

In [ ]:
#Reads nd2 files from nsparc, it also works for tf files
#Returns an ND2reader object
def readDataset(path, file):
    
    datafile = os.path.join(path, file)
    print(datafile)
    os.chdir(path)
    print(os.getcwd())
    
     #Load the .nd2 file (or tif)
    image_raw = pims.open(datafile)
    return image_raw

# image_raw = readDataset(path, file)

# print("Type:", type(image_raw)) 
# print("\nfeatures:\n", image_raw, "\n")

In [ ]:
#Extracts a channel for big fish analysis and returns a np array
#Takes an N2reader object and turns it into an np.ndarray
def getChannel(image_raw, ch):
    
    if ch == ch1:
        print("DAPI analysis")
        channel = 0
    elif ch == ch2:
        print("AF488 analysis")
        channel = 1
    elif ch == ch3:
        print("Cy3 analysis")
        channel = 2
    elif ch == ch4:
        print("Cy5 analysis")
        channel = 3

    number_of_Z = len(image_raw)
    array_list = []
    for z_slice in range(number_of_Z):
        
        frame = image_raw.get_frame_2D(c=channel, t=0, z=z_slice) #iterate t if needed
        array_list.append(frame)
        #print(type(frame))
        
    channel_stack = np.stack(array_list)
    #print(type(channel_stack))
    print(type(channel_stack), channel_stack.shape)
    return (channel_stack)

# cy3_image = getChannel(image_raw, ch3)
# cy5_image = getChannel(image_raw, ch4)

In [ ]:
#inputs cell pose masks read by pims, and return a numpy array
def cpmasks2numpy(cp_masks_tif):
    
    print("cell pose masks analysis")
    number_of_Z = len(cp_masks_tif)
    array_list = []
    for z_slice in range(number_of_Z):
        
        frame = cp_masks_tif[z_slice] #iterate t if needed
        array_list.append(frame)
        #print(type(frame))
        
    cpmasks_stack = np.stack(array_list)
    print(type(cpmasks_stack), cpmasks_stack.shape)
    
    return(cpmasks_stack)

In [ ]:
#Plots all stacks in a np.stack array
def displayNPStack(channel_stack):
    
    #adjust contrast
    vmin = np.percentile(channel_stack, 0)
    vmax = np.percentile(channel_stack, 100)
    
    for z_slice in range(len(channel_stack)):
        print("Slice ", z_slice)
        plt.figure(figsize=(10, 10))
        plt.imshow(channel_stack[z_slice], cmap='gray', vmin=vmin, vmax=vmax)
        
        plt.show()
        #Close the figure explicitly to free up memory
        plt.close()
    
    return

#displayNPStack(cy3_image)

In [ ]:
#Plot maxZ projection
def displayMaxZproj(channel_stack_maxZ):

    # Display the projection
    plt.imshow(channel_stack_maxZ, cmap='gray')
    plt.title("Maximum Z Projection")
    plt.show()
    plt.close()
    
    return()

In [ ]:
#Convert nm to pixels, input nm to convert and pixel size
#output distance equivalence in pixels
def nm2px(nms, z_pxsize, y_pxsize, x_pxsize):
    
    nm_in_pixels_z = np.round(nms / z_pxsize)
    nm_in_pixels_y = np.round(nms / y_pxsize)
    nm_in_pixels_x = np.round(nms / x_pxsize)
    
    return(nm_in_pixels_z, nm_in_pixels_y, nm_in_pixels_x)

In [ ]:
#Efficiently compute coordinates of voxels inside an ellipsoid, clipped within the volume bounds.
    
#   Parameters:
#   shape : tuple (z, y, x)
#   center : tuple (cz, cy, cx)
#   radii : tuple (rz, ry, rx)
    
#   Returns:
#   ellipsoid_mask : binary np array with mask shape
#   coords_inside_ellipsoid : ndarray (N, 3)

def ellipsoid_coords_zyx(image_shape, center_coords, radii):

    # Calculate bounding box around ellipsoid, clipped by volume bounds
    z_min = max(int(np.floor(center_coords[0] - radii[0])), 0)
    z_max = min(int(np.ceil(center_coords[0] + radii[0])) + 1, image_shape[0])
    y_min = max(int(np.floor(center_coords[1] - radii[1])), 0)
    y_max = min(int(np.ceil(center_coords[1] + radii[1])) + 1, image_shape[1])
    x_min = max(int(np.floor(center_coords[2] - radii[2])), 0)
    x_max = min(int(np.ceil(center_coords[2] + radii[2])) + 1, image_shape[2])
    
    # Create smaller coordinate arrays within bounding box
    z_range = np.arange(z_min, z_max)
    y_range = np.arange(y_min, y_max)
    x_range = np.arange(x_min, x_max)
    
    #Create meshes to iterated through z, y, and x (mathematician trick to not use for loops)
    z_mesh, y_mesh, x_mesh = np.meshgrid(z_range, y_range, x_range, indexing='ij')
    
    #Ellipsoid equation to find pixels inside ellipse
    ellipsoid_eq = (((z_mesh - center_coords[0]) / radii[0]) ** 2 +
                    ((y_mesh - center_coords[1]) / radii[1]) ** 2 +
                    ((x_mesh - center_coords[2]) / radii[2]) ** 2)
    
    ellipsoid_mask = ellipsoid_eq <= 1
    
    coords_inside_ellipsoid = np.argwhere(ellipsoid_mask)
    
    # coords_inside is relative to bounding box — convert back to full volume coords
    coords_inside_ellipsoid[:, 0] = coords_inside_ellipsoid[:, 0] + z_min
    coords_inside_ellipsoid[:, 1] = coords_inside_ellipsoid[:, 1] + y_min
    coords_inside_ellipsoid[:, 2] = coords_inside_ellipsoid[:, 2] + x_min
    
    # turn boolean mask into binary mask for output
    ellipsoid_mask = ellipsoid_mask.astype(int)
    
    return ellipsoid_mask, coords_inside_ellipsoid

###Test

# z_nm_px, y_nm_px, x_nm_px = nm2px(500, 195, 73, 73)

# print(z_nm_px)
# print(y_nm_px)
# print(x_nm_px)

# image_shape = af488_image.shape
# center_coords = (6, 1026, 1398)
# radii = (z_nm_px, y_nm_px, x_nm_px)

# print("image_shape", image_shape)
# print("center_coords", center_coords)
# print("radii", radii)

# print("image_shape", image_shape)
# print("center_coords", center_coords)
# print("radii", radii)

# #Now try  inputs:
# ellipsoid_mask, coords_inside_ellipsoid = ellipsoid_coords_zyx(image_shape, center_coords, radii)

# print("min z value:", np.min(coords_inside_ellipsoid[:,0]))
# print("max z value:", np.max(coords_inside_ellipsoid[:,0]))

# print("min y value:", np.min(coords_inside_ellipsoid[:,1]))
# print("max y value:", np.max(coords_inside_ellipsoid[:,1]))

# print("min x value:", np.min(coords_inside_ellipsoid[:,2]))
# print("max x value:", np.max(coords_inside_ellipsoid[:,2]))

# #plot ellipsoid
# displayNPStack(ellipsoid_mask.astype(int))
# boolean_image_maxZ = np.max(ellipsoid_mask, axis=0) #get maxZ projection
# displayMaxZproj(boolean_image_maxZ)


In [ ]:
#takes coordinates of the 3d ellipsoid and convert them into line collections (a contour) that can be plotted in 2d
def coor2contours2lc(coords_inside_ellipsoid, color, linewidths):
    ######
    z_coor, y_coor, x_coor = coords_inside_ellipsoid[:, 0], coords_inside_ellipsoid[:, 1], coords_inside_ellipsoid[:, 2]

    # Size of the 3D mask:
    z_size = z_coor.max() + 1
    y_size = y_coor.max() + 1
    x_size = x_coor.max() + 1

    mask_3d = np.zeros((z_size, y_size, x_size), dtype=np.uint8)
    mask_3d[z_coor, y_coor, x_coor] = 1
        
    mask_2d_maxproj = mask_3d.max(axis=0)
    contours = measure.find_contours(mask_2d_maxproj, level=0.5)  # for max projection
        
    contour_lines = [np.flip(cnt, axis=1) for cnt in contours]

    # Create the collection
    lc = LineCollection(contour_lines, colors = color, linewidths = linewidths)
        
    return(lc)

In [ ]:
#input, two 2d images
#output, overlayed image in magenta and yellow
def overlayImages(image1, image2):
    
    #Example grayscale images (same shape)
    #image1 Will appear magenta
    #image2 Will appear yellow

    # Normalize to [0, 1]
    image1 = (image1 - image1.min()) / (image1.max() - image1.min())
    image2 = (image2 - image2.min()) / (image2.max() - image2.min())

    # Create RGB composite
    overlay = np.zeros((image1.shape[0], image1.shape[1], 3))

    # Magenta = Red + Blue
    overlay[..., 0] = image1  # Red channel
    overlay[..., 2] = image1  # Blue channel

    # Yellow = Red + Green
    overlay[..., 0] += image2  # Add to Red channel
    overlay[..., 1] += image2  # Green channel

    # Clip values to [0, 1]
    overlay = np.clip(overlay, 0, 1)
    
    return(overlay)

In [ ]:
#Read channels in image
image_raw = readDataset(input_path, input_file)
cp_masks_tif = readDataset(mask_path, mask_file)

print("Type:", type(image_raw)) 
print("\nfeatures:\n", image_raw, "\n")

print("Type:", type(cp_masks_tif)) 
print("\nfeatures:\n", cp_masks_tif, "\n")

#convert raw images to np arrays

dapi_image = getChannel(image_raw, ch1)
af488_image = getChannel(image_raw, ch2)
cy3_image = getChannel(image_raw, ch3)
cy5_image = getChannel(image_raw, ch4)

cp_masks = cpmasks2numpy(cp_masks_tif)


In [ ]:
#Plot max Z projections of all channels
dapi_image_maxZ = np.max(dapi_image, axis=0) #get maxZ projection
displayMaxZproj(dapi_image_maxZ) #display the projection

af488_image_maxZ = np.max(af488_image, axis=0) #get maxZ projection
displayMaxZproj(af488_image_maxZ) #display the projection

cy3_image_maxZ = np.max(cy3_image, axis=0) #get maxZ projection
displayMaxZproj(cy3_image_maxZ) #display the projection

cy5_image_maxZ = np.max(cy5_image, axis=0) #get maxZ projection
displayMaxZproj(cy5_image_maxZ) #display the projection

cp_masks_maxZ = np.max(cp_masks, axis=0) #get maxZ projection
displayMaxZproj(cp_masks_maxZ) #display the projection

In [ ]:
# #Iterate through masks to assess masking

# mask_numbers = np.unique(cp_masks)
# for m in mask_numbers:
#     print("mask analyzed:", m)
    
#     cp_masks_copy = cp_masks.copy()
#     cp_masks_copy[cp_masks != m] = 0
#     cp_masks_copy[cp_masks == m] = 1
    
#     #plot masks
#     cp_masks_copy_maxZ = np.max(cp_masks_copy, axis=0) #get maxZ projection
#     displayMaxZproj(cp_masks_copy_maxZ) #display the projection
    
#     #plot dapi channel masked
# #     dapi_masked = dapi_image * cp_masks_copy
# #     dapi_masked_maxZ = np.max(dapi_masked, axis=0) #get maxZ projection
# #     displayMaxZproj(dapi_masked_maxZ) #display the projection
    
#     #plot af488 channel masked
#     af488_masked = af488_image * cp_masks_copy
#     af488_masked_maxZ = np.max(af488_masked, axis=0) #get maxZ projection
#     displayMaxZproj(af488_masked_maxZ) #display the projection
    
# print("Mask plotting finished")


In [ ]:
#Iterate through masks to obtain the mean fluorescence intensity of each mask

cellmask_numbers = np.unique(cp_masks)
cellmask_expression_list = []
for m in cellmask_numbers:
    
    coords_cellmask = np.argwhere(cp_masks == m)
    intensityValues_cellmask = af488_image[coords_cellmask[:, 0], coords_cellmask[:, 1], coords_cellmask[:, 2]]
    mean_expression_cellmask = np.mean(intensityValues_cellmask)
    median_expression_cellmask = np.median(intensityValues_cellmask)

    cellmask_expression_list.append([m, mean_expression_cellmask, median_expression_cellmask])
    
    print("mask analyzed:", m, "mean:", mean_expression_cellmask, "median:", median_expression_cellmask)
    
cellmask_expression_array = np.array(cellmask_expression_list)
#print(cellmask_expression_array)

In [ ]:
fishPairs = os.path.join(fishPairs_path, fishPairs_file)
print(fishPairs)
spots_df = pd.read_csv(fishPairs, delimiter='\t')
print(spots_df)

#Assign spots to cell masks
spots_in_masks_list = []
for index, row in spots_df.iterrows():
    
    sp_id = row[0]
    sp_z = row[7]
    sp_y = row[8]
    sp_x = row[9]
    
    sp_mask = cp_masks[sp_z, sp_y, sp_x]
    
    #spotid #z #y #x #sp_mask
    spots_in_masks_list.append([sp_id, sp_z, sp_y, sp_x, sp_mask])
    
    print("spot:", row[0], "is in", sp_mask) 

spots_in_masks_array = np.array(spots_in_masks_list)
#print(spots_in_masks_array)

In [ ]:
#Check intensity of voxels within ellipsoids around each spot

af488_image_shape = af488_image.shape
nm_radius_list = [100, 200, 300, 400, 800, 1000]
z_voxel = 195
y_voxel = 73
x_voxel = 73

#list to save
#spotid #z #y #x #cell_mask #expression #300nm #600nm #900nm #1200nm
spot_cellmask_intensity_list = []

for sp_info in spots_in_masks_array:
    #print("spot info", sp_info)
    center_coords = (sp_info[1], sp_info[2], sp_info[3])
    
    #iterate through a series of radii for different ellipsoid sizes and save mean values for each ellipsoid
    mean_intensity_ellipse_list = []
    max_intensity_ellipse_list = []
    
    mean_intensity_ellipse_neg_control_list = []
    max_intensity_ellipse_neg_control_list = []
    
    pp_image_values_ellipse_list = []
    pp_image_values_ellipse_neg_control_list = []

    for nm_radius in nm_radius_list:
        #print(nm_radius)
        
        #convert voxel dimentions in nanometers to pixels
        nm_px_radii = nm2px(nm_radius, z_voxel, y_voxel, x_voxel)
        #print(nm_radius)
        #print(nm_px_radii)
        
        #obtain ellipsoid masks and coords of different sizes
        ellipsoid_mask, coords_inside_ellipsoid = ellipsoid_coords_zyx(af488_image_shape, center_coords, nm_px_radii)
        
        #print shape of coordinates arraids for ellipses of different sizes
        #print(nm_radius, "nm ellipse shape:", coords_inside_ellipsoid.shape)
        
        #Split ellipse coordinates into separate arrays
        z_cie, y_cie, x_cie = coords_inside_ellipsoid[:, 0], coords_inside_ellipsoid[:, 1], coords_inside_ellipsoid[:, 2]
        
        #fetch values from af488_image that correspond to ellipse coordinates
        image_values_ellipse = af488_image[z_cie, y_cie, x_cie]
            
        #calculate mean intensity of af488_image voxels inside the ellipse and save in a list
        mean_intensity_ellipse = image_values_ellipse.mean()
        mean_intensity_ellipse_list.append(mean_intensity_ellipse)
        
        #calculate max intensity of af488_image voxels inside the ellipse and save in a list
        max_intensity_ellipse = image_values_ellipse.max()
        max_intensity_ellipse_list.append(max_intensity_ellipse)
        
        #calculate mean intensity and max intensity of DAPI voxels inside ellipse as negative control
        image_values_ellipse_neg_control = dapi_image[z_cie, y_cie, x_cie]
        
        mean_intensity_ellipse_neg_control = image_values_ellipse_neg_control.mean()
        mean_intensity_ellipse_neg_control_list.append(mean_intensity_ellipse_neg_control)
        
        max_intensity_ellipse_neg_control = image_values_ellipse_neg_control.max()
        max_intensity_ellipse_neg_control_list.append(max_intensity_ellipse_neg_control)
        
        #calculate percentile point for lower 20%, ellipse and control
        pp_image_values_ellipse = np.percentile(image_values_ellipse, 20)
        pp_image_values_ellipse_list.append(pp_image_values_ellipse)
        
        pp_image_values_ellipse_neg_control = np.percentile(image_values_ellipse_neg_control, 20)
        pp_image_values_ellipse_neg_control_list.append(pp_image_values_ellipse_neg_control)
        ######
        
        #print(nm_radius, "nm ellipse intensity:", mean_intensity_ellipse)
        
    #find which cell mask row corresponds with the cell mask of current FISH spot
    cellmask_expression_index = np.where(cellmask_expression_array[:, 0] == sp_info[4])[0]
    cell_mask_exp = cellmask_expression_array[cellmask_expression_index, :]
    cell_mask_exp = cell_mask_exp.flatten()
    #print(cell_mask_exp, sp_info)
    #print(mean_intensity_ellipse_list)
    
    #output list
    spot_cellmask_intensity_list.append([sp_info[0], sp_info[1], sp_info[2], sp_info[3],  sp_info[4], 
                                        cell_mask_exp[1], cell_mask_exp[2],
                                         
                                        mean_intensity_ellipse_list[0], mean_intensity_ellipse_list[1], 
                                        mean_intensity_ellipse_list[2], mean_intensity_ellipse_list[3],
                                        mean_intensity_ellipse_list[4], mean_intensity_ellipse_list[5],
                                         
                                        max_intensity_ellipse_list[0], max_intensity_ellipse_list[1], 
                                        max_intensity_ellipse_list[2], max_intensity_ellipse_list[3],
                                        max_intensity_ellipse_list[4], max_intensity_ellipse_list[5],
                                         
                                        mean_intensity_ellipse_neg_control_list[0], mean_intensity_ellipse_neg_control_list[1],
                                        mean_intensity_ellipse_neg_control_list[2], mean_intensity_ellipse_neg_control_list[3],
                                        mean_intensity_ellipse_neg_control_list[4], mean_intensity_ellipse_neg_control_list[5],
                                         
                                        max_intensity_ellipse_neg_control_list[0], max_intensity_ellipse_neg_control_list[1],
                                        max_intensity_ellipse_neg_control_list[2], max_intensity_ellipse_neg_control_list[3],
                                        max_intensity_ellipse_neg_control_list[4], max_intensity_ellipse_neg_control_list[5], 
                                        
                                        pp_image_values_ellipse_list[0], pp_image_values_ellipse_list[1],
                                        pp_image_values_ellipse_list[2], pp_image_values_ellipse_list[3],
                                        pp_image_values_ellipse_list[4], pp_image_values_ellipse_list[5],
                                         
                                        pp_image_values_ellipse_neg_control_list[0], pp_image_values_ellipse_neg_control_list[1],
                                        pp_image_values_ellipse_neg_control_list[2], pp_image_values_ellipse_neg_control_list[3], 
                                        pp_image_values_ellipse_neg_control_list[4], pp_image_values_ellipse_neg_control_list[5]
                                         
                                        ])

spot_cellmask_intensity_array = np.array(spot_cellmask_intensity_list)
print("spot_cellmask_intensity_array saved")
#print(spot_cellmask_intensity_array)

In [ ]:
#Iterate over all processed spots to assess if cell mask segmentation and overall analysis was correct
#And make nice plots of results as sanity check

vmin_nuc = np.percentile(dapi_image, 40) #set contrast
vmax_nuc = np.percentile(dapi_image, 99.9) #set contrast

vmin_af488 = np.percentile(af488_image, 30) #set contrast
vmax_af488 = np.percentile(af488_image, 99) #set contrast

vmin_cy3 = np.percentile(cy3_image, 60) #set contrast
vmax_cy3 = np.percentile(cy3_image, 99.99) #set contrast

vmin_cy5 = np.percentile(cy5_image, 60) #set contrast
vmax_cy5 = np.percentile(cy5_image, 99.99) #set contrast

output_data = []

for p_sp in spot_cellmask_intensity_array:
    #print(p_sp)
    
    #information from analysis to plot
    sp_id = p_sp[0].astype(int)
    
    sp_z = p_sp[1].astype(int)
    sp_y = p_sp[2].astype(int)
    sp_x = p_sp[3].astype(int)
    
    cm_id = p_sp[4].astype(int)
    cell_mean = p_sp[5]
    cell_median = p_sp[6]
    radii_string = str(nm_radius_list[0]) + "; " + str(nm_radius_list[1]) + "; " + str(nm_radius_list[2]) + "; " + str(nm_radius_list[3]) + "; " + str(nm_radius_list[4]) + "; " + str(nm_radius_list[5])
        
    #plot display specifics
    window_size = 200
    x_low = sp_x - window_size
    x_high = sp_x + window_size
    y_low = sp_y - window_size
    y_high = sp_y + window_size
    #y_arrow = sp_y - 12

    x_low = np.maximum(x_low, 0)
    x_high = np.minimum(x_high, 2047)
    y_low = np.maximum(y_low, 0)
    y_high = np.minimum(y_high, 2047)
    #y_arrow = np.maximum(y_arrow, 0)
    
    #plot display specifics - zoom
    window_size_zoom = 15
    x_low_zoom = sp_x - window_size_zoom
    x_high_zoom = sp_x + window_size_zoom
    y_low_zoom = sp_y - window_size_zoom
    y_high_zoom = sp_y + window_size_zoom

    x_low_zoom = np.maximum(x_low_zoom, 0)
    x_high_zoom = np.minimum(x_high_zoom, 2047)
    y_low_zoom = np.maximum(y_low_zoom, 0)
    y_high_zoom = np.minimum(y_high_zoom, 2047)
    
    if cm_id == 0:
        continue
        
    ##set figure layout
    fig, (ax1, ax2, ax3, ax4) = plt.subplots(1, 4, figsize=(20, 5), gridspec_kw={'width_ratios': [1, 1, 1, 1]})
    
    fig.suptitle(output_plot_folder_name + "\nmask id: " + str(cm_id) +
                 "\nspot id: " + str(sp_id), x=0.1, y=1.05, ha='left', fontsize=14)
    
    ##plot dapi
    ax1.imshow(dapi_image[sp_z], origin='upper', cmap = 'gray', vmin = vmin_nuc, vmax = vmax_nuc)
    
    #process cellmasks and retrieve contours
    color_contour = 'white'
    linewidth = 1 
    
    coords_cellmask = np.argwhere(cp_masks == cm_id)
    filtered_coords_cellmask = coords_cellmask[coords_cellmask[:, 0] == sp_z]
    #print(filtered_coords_cellmask)

    line_collections_cellmask_contour = coor2contours2lc(filtered_coords_cellmask, color_contour, linewidth)
    ax1.add_collection(line_collections_cellmask_contour)
    #
    
    ax1.scatter(sp_x, sp_y, marker='o', color = "b", linewidth=5, alpha = 1, s = 20)
    ax1.set_xlim(x_low, x_high) #zoom to mask currently analyzed
    ax1.set_ylim(y_low, y_high)
    ax1.set_ylim(ax1.get_ylim()[::-1])  # Reverse the y-axis limits
    ax1.set_title("DAPI")
    
    
    ##plot af488 with ellipsoids
    ax2.imshow(af488_image[sp_z], origin='upper', cmap = 'gray', vmin = vmin_af488, vmax = vmax_af488)
    ax2.scatter(sp_x, sp_y, marker='o', color = "r", linewidth=5, alpha = 1, s = 20)

    #process cellmasks and retrieve contours
    color_contour = 'white'
    linewidth = 1 
    
    coords_cellmask = np.argwhere(cp_masks == cm_id)
    filtered_coords_cellmask = coords_cellmask[coords_cellmask[:, 0] == sp_z]
    #print(filtered_coords_cellmask)

    line_collections_cellmask_contour = coor2contours2lc(filtered_coords_cellmask, color_contour, linewidth)
    ax2.add_collection(line_collections_cellmask_contour)
    #
    
    ax2.set_xlim(x_low, x_high) #zoom to mask currently analyzed
    ax2.set_ylim(y_low, y_high)
    ax2.set_ylim(ax2.get_ylim()[::-1])  # Reverse the y-axis limits
    ax2.set_title("af488 image\nmean: " + "{:.2f}".format(cell_mean) + "; median: " + "{:.2f}".format(cell_median))
    
    
    ##plot cy5 - cy3
    cy5cy3_overlay = overlayImages(cy5_image[sp_z], cy3_image[sp_z]) #image1 magenta, image2 yellow
    ax3.imshow(cy5cy3_overlay, origin='upper', cmap = 'gray', vmin = min(vmin_cy3, vmin_cy5), vmax = max(vmax_cy3, vmax_cy5))
    ax3.set_xlim(x_low_zoom, x_high_zoom) #zoom to mask currently analyzed
    ax3.set_ylim(y_low_zoom, y_high_zoom)
    ax3.set_ylim(ax3.get_ylim()[::-1])  # Reverse the y-axis limits
    ax3.set_title("Cy5 magenta - Cy3 yellow\nz:" + str(sp_z) + " - y:" + str(sp_y) + " - x:" + str(sp_x))
    
    
    ##plot condensates zoom
    ax4.imshow(af488_image[sp_z], origin='upper', cmap = 'gray', vmin = vmin_af488, vmax = vmax_af488)
    ax4.set_xlim(x_low_zoom, x_high_zoom) #zoom to mask currently analyzed
    ax4.set_ylim(y_low_zoom, y_high_zoom)
    ax4.set_ylim(ax4.get_ylim()[::-1])  # Reverse the y-axis limits
    ax4.set_title("Cy5")
    #process ellipsoids and retrieve contours
    for nm_radius in nm_radius_list:
        center_coords = (sp_z, sp_y, sp_x)
        nm_px_radii = nm2px(nm_radius, z_voxel, y_voxel, x_voxel)
        ellipsoid_mask, coords_inside_ellipsoid = ellipsoid_coords_zyx(af488_image_shape, center_coords, nm_px_radii)       
        #print(coords_inside_ellipsoid)
        
        color_contour = 'red'
        linewidth = 1
        line_collections_contour = coor2contours2lc(coords_inside_ellipsoid, color_contour, linewidth)
        ax4.add_collection(line_collections_contour)
        
    ax4.set_title("af488 - radii (nm): pp20; max intensity (au)" + "\n" +
                str(nm_radius_list[0]) + ": {:.2f}; ".format(p_sp[31]) + "{:.2f} -- ".format(p_sp[13]) +
                str(nm_radius_list[1]) + ": {:.2f}; ".format(p_sp[32]) + "{:.2f}\n".format(p_sp[14]) +
                str(nm_radius_list[2]) + ": {:.2f}; ".format(p_sp[33]) + "{:.2f} -- ".format(p_sp[15]) +
                str(nm_radius_list[3]) + ": {:.2f}; ".format(p_sp[34]) + "{:.2f} -- ".format(p_sp[16]) +
                str(nm_radius_list[4]) + ": {:.2f}; ".format(p_sp[35]) + "{:.2f}\n".format(p_sp[17]) +
                str(nm_radius_list[5]) + ": {:.2f}; ".format(p_sp[36]) + "{:.2f}".format(p_sp[18])
                 )
        #
    
    plt.show()
    
    ask_radial_analysis = "Is this result good for further analysis (y/n)\n"
    answer_radial_analysis = input(ask_radial_analysis)
        
    if answer_radial_analysis == 'y':
        
        output_data.append(p_sp) #append chosen data points to output data
        
        image_result_file_name = output_plot_folder_name + "_m" + str(cm_id) + "_s" + str(sp_id) + ".png"
        image_result_file = output_plot_folder + "/" + image_result_file_name
        fig.savefig(image_result_file, bbox_inches='tight') #save image
        
    plt.close(fig)
        
print("analysis finished\nplot files and output_data generated")

#print(output_data)
    

In [ ]:
print(len(output_data), "spots were selected out of", len(spot_cellmask_intensity_array))

#print(output_data)
#convert selected_data toy pandas data frame and save as a .txt file

#spotid #z #y #x #cell_mask #expression #300nm #600nm #900nm #1200nm
#print(nm_radius_list)

headers = [
    'spot_id', 'z', 'y', 'x', 
    'cellmask_id', 'expmean', 'expmedian',
    str(nm_radius_list[0])+"mean", str(nm_radius_list[1])+"mean", str(nm_radius_list[2])+"mean", str(nm_radius_list[3])+"mean",
    str(nm_radius_list[4])+"mean", str(nm_radius_list[5])+"mean",
    
    str(nm_radius_list[0])+"max", str(nm_radius_list[1])+"max", str(nm_radius_list[2])+"max", str(nm_radius_list[3])+"max",
    str(nm_radius_list[4])+"max", str(nm_radius_list[5])+"max",
    
    str(nm_radius_list[0])+"dapimean", str(nm_radius_list[1])+"dapimean", str(nm_radius_list[2])+"dapimean", str(nm_radius_list[3])+"dapimean",
    str(nm_radius_list[4])+"dapimean", str(nm_radius_list[5])+"dapimean",
    
    str(nm_radius_list[0])+"dapimax", str(nm_radius_list[1])+"dapimax", str(nm_radius_list[2])+"dapimax", str(nm_radius_list[3])+"dapimax",
    str(nm_radius_list[4])+"dapimax", str(nm_radius_list[5])+"dapimax",
    
    str(nm_radius_list[0])+"perc20", str(nm_radius_list[1])+"perc20", str(nm_radius_list[2])+"perc20", str(nm_radius_list[3])+"perc20",
    str(nm_radius_list[4])+"perc20", str(nm_radius_list[5])+"perc20",
    
    str(nm_radius_list[0])+"dapiperc20", str(nm_radius_list[1])+"dapiperc20", str(nm_radius_list[2])+"dapiperc20", str(nm_radius_list[3])+"dapiperc20",
    str(nm_radius_list[4])+"dapiperc20", str(nm_radius_list[5])+"dapiperc20",
]

output_df = pd.DataFrame(output_data, columns=headers)
print(output_df, "\n")

# Save to file
output_df.to_csv(output_file, sep='\t', index=True, index_label="id")
print('data saved in:', output_file)

In [ ]:
## Generate a supplement file with the parameters used in the current analysis

current_date = datetime.now() #Obtain date
formatted_date = current_date.strftime("%Y-%m-%d")
formatted_date_parameters = "Analysis date: "  + formatted_date + "\n"
input_date_parameters = "input_date: " + str(input_path.split('/')[-2]) + "\n"

input_file_parameters = "input_file: " + str(input_file) + "\n"
mask_file_parameters = "mask_file: " + str(mask_file) + "\n"
fishPairs_file_parameters = "fishPairs: " + str(fishPairs_file) + "\n"

voxel_size_parameters = "voxel size zyx (nm): " + str(z_voxel) + "; " + str(y_voxel) + "; " + str(x_voxel) + "\n"
ellipsoid_size_parameters = "ellipsoid sizes (nm): " + str(nm_radius_list) + "\n"

spot_masks_detected_parameters = "spot_masks_detected: " + str(len(spot_cellmask_intensity_array)) + "\n"
spot_masks_selected_parameters = "spot_masks_selected: " + str(len(output_data)) + "\n"


print(formatted_date_parameters)
print(input_date_parameters)
print(input_file_parameters)
print(mask_file_parameters)
print(fishPairs_file_parameters)
print(voxel_size_parameters)
print(ellipsoid_size_parameters)
print(spot_masks_detected_parameters)
print(spot_masks_selected_parameters)

try:
    with open(output_file_supplement, 'w') as f:
        
        f.write(formatted_date_parameters)
        f.write(input_date_parameters)
        f.write(input_file_parameters)
        f.write(mask_file_parameters)
        f.write(fishPairs_file_parameters)
        f.write(voxel_size_parameters)
        f.write(ellipsoid_size_parameters)
        f.write(spot_masks_detected_parameters)
        f.write(spot_masks_selected_parameters)
        
except IOError:
    print("Could not open file {output_file_supplement} for writing") #Handle the case where the file cannot be opened or written to

print("File:\n", output_file_supplement, "\ngenerated")